In [1]:
# =========================================================
# AUTOMATIZACIÓN NASCA ENEL - DESCARGA Y CONSOLIDACIÓN
# =========================================================

import os
import re
import time
import shutil
import traceback
from pathlib import Path

import pandas as pd
import numpy as np

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException


# =========================================================
# CONFIGURACIÓN GENERAL
# =========================================================

URL_LOGIN = "https://nasca.enel.com/app-web-synergia/#/consulta-cuenta"

USUARIO = "pe15733437"
CONTRASENA = "Cds1987200523@"   # <-- REEMPLAZAZA

RUTA_EXCEL_BASE = r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\BUSCADOR DE COMPENSACIONES OFICIO\IMPUT\SUMINISTROS.xlsx"

RUTA_OUTPUT = r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\BUSCADOR DE COMPENSACIONES OFICIO\OUT-PUT"
RUTA_DESCARGAS = os.path.join(RUTA_OUTPUT, "DESCARGAS_TEMP")
RUTA_INDIVIDUALES = os.path.join(RUTA_OUTPUT, "EXCEL_INDIVIDUALES")
RUTA_LOGS = os.path.join(RUTA_OUTPUT, "LOGS")
RUTA_CAPTURAS = os.path.join(RUTA_OUTPUT, "CAPTURAS")

RUTA_CONSOLIDADO = os.path.join(RUTA_OUTPUT, "CONSOLIDADO_COMPENSACIONES.xlsx")
RUTA_RESUMEN = os.path.join(RUTA_OUTPUT, "RESUMEN_PROCESO.xlsx")
RUTA_LOG_ERRORES = os.path.join(RUTA_LOGS, "errores_detalle.txt")

TIMEOUT = 25
TIEMPO_DESCARGA_MAX = 60

MODO_HEADLESS = False
CHROMEDRIVER_PATH = None
# CHROMEDRIVER_PATH = r"C:\ruta\chromedriver.exe"


# =========================================================
# CREAR CARPETAS
# =========================================================

for ruta in [RUTA_OUTPUT, RUTA_DESCARGAS, RUTA_INDIVIDUALES, RUTA_LOGS, RUTA_CAPTURAS]:
    os.makedirs(ruta, exist_ok=True)


# =========================================================
# FUNCIONES AUXILIARES
# =========================================================

def limpiar_nombre_archivo(texto: str) -> str:
    texto = str(texto).strip()
    texto = re.sub(r'[<>:"/\\|?*]+', '_', texto)
    texto = re.sub(r'\s+', '_', texto)
    return texto


def convertir_numero(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip()

    if valor == "":
        return np.nan

    valor = valor.replace(" ", "")

    if "." in valor and "," in valor:
        valor = valor.replace(".", "").replace(",", ".")
    else:
        if "," in valor:
            valor = valor.replace(",", ".")

    try:
        return float(valor)
    except:
        return np.nan


def esperar_archivo_descargado(carpeta_descargas, timeout=60, prefijo="Reporte"):
    inicio = time.time()

    while time.time() - inicio < timeout:
        archivos = list(Path(carpeta_descargas).glob(f"{prefijo}*"))

        archivos_validos = [
            a for a in archivos
            if not str(a).endswith(".crdownload") and not str(a).endswith(".tmp")
        ]

        if archivos_validos:
            archivo_reciente = max(archivos_validos, key=lambda x: x.stat().st_mtime)

            size_1 = archivo_reciente.stat().st_size
            time.sleep(1.5)
            size_2 = archivo_reciente.stat().st_size

            if size_1 == size_2 and size_2 > 0:
                return str(archivo_reciente)

        time.sleep(1)

    return None


def eliminar_reportes_previos(carpeta_descargas, prefijo="Reporte"):
    archivos = list(Path(carpeta_descargas).glob(f"{prefijo}*"))
    for archivo in archivos:
        try:
            os.remove(archivo)
        except:
            pass


def iniciar_driver():
    chrome_options = Options()

    prefs = {
        "download.default_directory": RUTA_DESCARGAS,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
        "plugins.always_open_pdf_externally": True,
    }
    chrome_options.add_experimental_option("prefs", prefs)

    if MODO_HEADLESS:
        chrome_options.add_argument("--headless=new")

    chrome_options.add_argument("--start-maximized")
    chrome_options.add_argument("--disable-notifications")
    chrome_options.add_argument("--disable-popup-blocking")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    if CHROMEDRIVER_PATH:
        service = Service(CHROMEDRIVER_PATH)
        driver = webdriver.Chrome(service=service, options=chrome_options)
    else:
        driver = webdriver.Chrome(options=chrome_options)

    return driver


def click_js(driver, element):
    driver.execute_script("arguments[0].click();", element)


def scroll_to_element(driver, element):
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", element)
    time.sleep(0.5)


def click_element_robusto(driver, by, locator, timeout=TIMEOUT):
    ultimo_error = None

    for _ in range(3):
        try:
            elem = WebDriverWait(driver, timeout).until(
                EC.presence_of_element_located((by, locator))
            )
            scroll_to_element(driver, elem)

            try:
                WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((by, locator))
                )
                elem.click()
            except:
                click_js(driver, elem)

            return elem

        except Exception as e:
            ultimo_error = e
            time.sleep(1)

    raise ultimo_error


def write_element(driver, by, locator, texto, clear_first=True, timeout=TIMEOUT):
    elem = WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((by, locator))
    )

    scroll_to_element(driver, elem)

    if clear_first:
        try:
            elem.clear()
        except:
            pass

    elem.send_keys(str(texto))
    return elem


def tomar_captura(driver, nombre):
    nombre = limpiar_nombre_archivo(nombre)
    ruta = os.path.join(RUTA_CAPTURAS, f"{nombre}.png")
    try:
        driver.save_screenshot(ruta)
    except:
        pass
    return ruta


def leer_excel_descargado(ruta_excel):
    ext = Path(ruta_excel).suffix.lower()

    if ext == ".xlsx":
        return pd.read_excel(ruta_excel, engine="openpyxl")
    elif ext == ".xls":
        try:
            return pd.read_excel(ruta_excel)
        except:
            return pd.read_excel(ruta_excel, engine="xlrd")
    else:
        return pd.read_excel(ruta_excel)


def agregar_columnas_y_guardar(
    ruta_archivo,
    interrupcion,
    suministro,
    periodo_anio_mes,
    compensacion_total,
    ruta_guardado_final
):
    df = leer_excel_descargado(ruta_archivo)

    df.columns = [str(c).strip() for c in df.columns]

    df["INTERRUPCION"] = interrupcion
    df["SUMINISTRO"] = str(suministro)
    df["PERIODO_ANIO_MES"] = str(periodo_anio_mes)

    comp = convertir_numero(compensacion_total)
    if pd.notna(comp):
        comp = -abs(comp)

    df["COMPENSACION_TOTAL"] = comp

    if "Valor" in df.columns:
        df["VALOR_NUM"] = df["Valor"].apply(convertir_numero)
    else:
        df["VALOR_NUM"] = np.nan

    if pd.notna(comp):
        df["COINCIDE_COMPENSACION"] = np.isclose(
            df["VALOR_NUM"].round(2),
            round(comp, 2),
            atol=0.01
        )
    else:
        df["COINCIDE_COMPENSACION"] = False

    df.to_excel(ruta_guardado_final, index=False)

    return df


def cambiar_a_nueva_ventana(driver, handles_antes, timeout=15):
    inicio = time.time()

    while time.time() - inicio < timeout:
        handles_actuales = driver.window_handles

        if len(handles_actuales) > len(handles_antes):
            nuevos = [h for h in handles_actuales if h not in handles_antes]
            if nuevos:
                driver.switch_to.window(nuevos[-1])
                time.sleep(2)
                return True

        time.sleep(1)

    return False


def asegurar_ventana_consulta_cuenta(driver, timeout=20):
    """
    Busca entre todas las ventanas una que tenga la ruta consulta-cuenta
    o que contenga el input del suministro.
    """
    inicio = time.time()

    while time.time() - inicio < timeout:
        handles = driver.window_handles

        for h in handles:
            try:
                driver.switch_to.window(h)
                time.sleep(1)

                url_actual = driver.current_url.lower()

                if "consulta-cuenta" in url_actual:
                    return True

                if len(driver.find_elements(
                    By.XPATH,
                    "/html/body/app-root/app-consulta-cuenta/div/div/div/div/div/div/div[3]/div/div/div[1]/div/div[2]/input"
                )) > 0:
                    return True

            except:
                pass

        time.sleep(1)

    return False


# =========================================================
# LOGIN Y NAVEGACIÓN
# =========================================================

def hacer_login(driver):
    driver.get(URL_LOGIN)
    time.sleep(3)

    usuario_xpath = "/html/body/app-root/app-login/div/div/div/div[2]/div[1]/form/div[1]/span/input"
    password_xpath = "/html/body/app-root/app-login/div/div/div/div[2]/div[1]/form/div[2]/span/input"

    write_element(driver, By.XPATH, usuario_xpath, USUARIO)
    write_element(driver, By.XPATH, password_xpath, CONTRASENA)

    driver.find_element(By.XPATH, password_xpath).send_keys(Keys.ENTER)

    WebDriverWait(driver, TIMEOUT).until(
        EC.presence_of_element_located((By.XPATH, "/html/body/app-root/app-layout"))
    )
    time.sleep(3)


def abrir_menu_consulta_cuenta(driver):
    click_element_robusto(
        driver,
        By.XPATH,
        "/html/body/app-root/app-layout/div/app-topbar/div/button/i"
    )
    time.sleep(1.5)

    click_element_robusto(
        driver,
        By.XPATH,
        "/html/body/app-root/app-layout/div/div[1]/app-sidebar/app-menu/ul/li[1]/ul/li/a/span[1]"
    )
    time.sleep(1.5)

    handles_antes = driver.window_handles.copy()

    # OJO: ahora clickeamos el <a>, no solo span[1]
    click_element_robusto(
        driver,
        By.XPATH,
        "/html/body/app-root/app-layout/div/div[1]/app-sidebar/app-menu/ul/li[1]/ul/li/ul/li/a"
    )
    time.sleep(3)

    se_abrio_nueva = cambiar_a_nueva_ventana(driver, handles_antes, timeout=10)

    if not se_abrio_nueva:
        # si no abrió nueva, igual buscamos entre handles o en la misma
        pass

    ok = asegurar_ventana_consulta_cuenta(driver, timeout=20)

    if not ok:
        tomar_captura(driver, "error_no_abre_consulta_cuenta")
        raise TimeoutException("No se logró ubicar la ventana/pestaña de consulta-cuenta.")


def buscar_suministro_en_consulta_cuenta(driver, suministro):
    campo_suministro_xpath = "/html/body/app-root/app-consulta-cuenta/div/div/div/div/div/div/div[3]/div/div/div[1]/div/div[2]/input"
    boton_buscar_xpath = "/html/body/app-root/app-consulta-cuenta/div[2]/div/div/div/div/div/div[3]/div/div/div[2]/div/div[1]/p-button/button/span[2]"

    campo = WebDriverWait(driver, TIMEOUT).until(
        EC.presence_of_element_located((By.XPATH, campo_suministro_xpath))
    )

    scroll_to_element(driver, campo)

    try:
        campo.click()
    except:
        pass

    try:
        campo.send_keys(Keys.CONTROL, "a")
        campo.send_keys(Keys.DELETE)
    except:
        pass

    try:
        campo.clear()
    except:
        pass

    campo.send_keys(str(suministro))
    time.sleep(1)

    try:
        campo.send_keys(Keys.ENTER)
        time.sleep(3)
    except:
        pass

    try:
        click_element_robusto(driver, By.XPATH, boton_buscar_xpath, timeout=8)
        time.sleep(3)
    except:
        pass


def abrir_consulta_facturacion(driver, suministro=None):
    boton_facturacion_xpath = "/html/body/app-root/app-consulta-cuenta/div[2]/div/div/div/div/div/div[1]/div/button[2]/img"

    ultimo_error = None

    for _ in range(3):
        try:
            boton = WebDriverWait(driver, TIMEOUT).until(
                EC.presence_of_element_located((By.XPATH, boton_facturacion_xpath))
            )

            scroll_to_element(driver, boton)

            try:
                boton.click()
            except:
                click_js(driver, boton)

            time.sleep(4)

            condiciones = [
                lambda d: "consulta-facturacion" in d.current_url.lower(),
                lambda d: len(d.find_elements(By.XPATH, '//*[@id="p-tabpanel-2-label"]/span[1]')) > 0,
                lambda d: len(d.find_elements(By.XPATH, '//*[@id="pr_id_3-table"]/tfoot/tr/td[2]/label[1]')) > 0,
            ]

            inicio = time.time()
            while time.time() - inicio < TIMEOUT:
                for condicion in condiciones:
                    try:
                        if condicion(driver):
                            time.sleep(2)
                            return True
                    except:
                        pass
                time.sleep(1)

            raise TimeoutException("No cargó consulta-facturacion luego del click.")

        except Exception as e:
            ultimo_error = e
            time.sleep(2)

    nombre_cap = f"error_abrir_facturacion_{suministro if suministro else 'sin_suministro'}"
    tomar_captura(driver, nombre_cap)
    raise ultimo_error


def descargar_excel_facturacion(driver):
    tab_xpath = '//*[@id="p-tabpanel-2-label"]/span[1]'
    descargar_xpath = '//*[@id="pr_id_3-table"]/tfoot/tr/td[2]/label[1]'

    inicio = time.time()
    ok = False

    while time.time() - inicio < TIMEOUT:
        if (
            len(driver.find_elements(By.XPATH, tab_xpath)) > 0
            or len(driver.find_elements(By.XPATH, descargar_xpath)) > 0
        ):
            ok = True
            break
        time.sleep(1)

    if not ok:
        raise TimeoutException("No se encontró la pestaña o botón de descarga en facturación.")

    try:
        click_element_robusto(driver, By.XPATH, tab_xpath, timeout=10)
        time.sleep(2)
    except:
        pass

    click_element_robusto(driver, By.XPATH, descargar_xpath, timeout=15)
    time.sleep(2)


def volver_a_consulta_cuenta(driver):
    """
    Vuelve a la ventana que contiene consulta-cuenta.
    Si no la encuentra, abre directamente la URL.
    """
    handles = driver.window_handles

    for h in handles:
        try:
            driver.switch_to.window(h)
            time.sleep(1)

            if "consulta-cuenta" in driver.current_url.lower():
                WebDriverWait(driver, TIMEOUT).until(
                    EC.presence_of_element_located((
                        By.XPATH,
                        "/html/body/app-root/app-consulta-cuenta/div/div/div/div/div/div/div[3]/div/div/div[1]/div/div[2]/input"
                    ))
                )
                return
        except:
            pass

    driver.get("https://nasca.enel.com/app-web-synergia/#/consulta-cuenta")
    time.sleep(4)

    WebDriverWait(driver, TIMEOUT).until(
        EC.presence_of_element_located((
            By.XPATH,
            "/html/body/app-root/app-consulta-cuenta/div/div/div/div/div/div/div[3]/div/div/div[1]/div/div[2]/input"
        ))
    )


# =========================================================
# PROCESAMIENTO PRINCIPAL
# =========================================================

def main():
    df_base = pd.read_excel(RUTA_EXCEL_BASE, dtype=str)
    df_base.columns = [str(c).strip().upper() for c in df_base.columns]

    columnas_requeridas = ["INTERRUPCION", "SUMINISTRO", "PERIODO_ANIO_MES", "COMPENSACION_TOTAL"]
    faltantes = [c for c in columnas_requeridas if c not in df_base.columns]

    if faltantes:
        raise ValueError(f"Faltan columnas en el Excel base: {faltantes}")

    df_base = df_base[columnas_requeridas].copy()
    df_base["COMPENSACION_TOTAL_NUM"] = df_base["COMPENSACION_TOTAL"].apply(convertir_numero)
    df_base["COMPENSACION_TOTAL_NEG"] = -df_base["COMPENSACION_TOTAL_NUM"].abs()

    resultados = []
    dataframes_ok = []

    driver = iniciar_driver()

    try:
        hacer_login(driver)
        abrir_menu_consulta_cuenta(driver)

        total = len(df_base)

        for i, row in df_base.iterrows():
            interrupcion = str(row["INTERRUPCION"]).strip()
            suministro = str(row["SUMINISTRO"]).strip()
            periodo = str(row["PERIODO_ANIO_MES"]).strip()
            comp_total_raw = row["COMPENSACION_TOTAL"]
            comp_total_neg = row["COMPENSACION_TOTAL_NEG"]

            nombre_archivo = limpiar_nombre_archivo(
                f"{interrupcion}_{periodo}_{suministro}.xlsx"
            )
            ruta_final_individual = os.path.join(RUTA_INDIVIDUALES, nombre_archivo)

            estado = "OK"
            observacion = ""
            ruta_archivo_final = ""
            filas_archivo = 0
            coincidencias = 0
            captura_error = ""

            print(f"[{i+1}/{total}] Procesando suministro: {suministro}")

            try:
                eliminar_reportes_previos(RUTA_DESCARGAS, prefijo="Reporte")

                volver_a_consulta_cuenta(driver)
                buscar_suministro_en_consulta_cuenta(driver, suministro)
                abrir_consulta_facturacion(driver, suministro=suministro)
                descargar_excel_facturacion(driver)

                ruta_descargada = esperar_archivo_descargado(
                    RUTA_DESCARGAS,
                    timeout=TIEMPO_DESCARGA_MAX,
                    prefijo="Reporte"
                )

                if not ruta_descargada:
                    raise FileNotFoundError("No se detectó la descarga del archivo Reporte.")

                if os.path.exists(ruta_final_individual):
                    os.remove(ruta_final_individual)

                shutil.move(ruta_descargada, ruta_final_individual)

                df_ind = agregar_columnas_y_guardar(
                    ruta_archivo=ruta_final_individual,
                    interrupcion=interrupcion,
                    suministro=suministro,
                    periodo_anio_mes=periodo,
                    compensacion_total=comp_total_raw,
                    ruta_guardado_final=ruta_final_individual
                )

                filas_archivo = len(df_ind)
                coincidencias = int(df_ind["COINCIDE_COMPENSACION"].sum()) if "COINCIDE_COMPENSACION" in df_ind.columns else 0
                ruta_archivo_final = ruta_final_individual

                dataframes_ok.append(df_ind)

                if coincidencias == 0:
                    estado = "SIN_COINCIDENCIA_VALOR"
                    observacion = "Se descargó archivo, pero ninguna fila de 'Valor' coincide con COMPENSACION_TOTAL negativa."

            except Exception as e:
                estado = "ERROR"
                observacion = str(e)
                captura_error = tomar_captura(driver, f"error_{suministro}_{interrupcion}")

                with open(RUTA_LOG_ERRORES, "a", encoding="utf-8") as f:
                    f.write("=" * 120 + "\n")
                    f.write(f"SUMINISTRO: {suministro}\n")
                    f.write(f"INTERRUPCION: {interrupcion}\n")
                    f.write(f"PERIODO_ANIO_MES: {periodo}\n")
                    f.write(f"ERROR: {str(e)}\n")
                    f.write(f"CAPTURA: {captura_error}\n")
                    f.write(traceback.format_exc())
                    f.write("\n")

            resultados.append({
                "INTERRUPCION": interrupcion,
                "SUMINISTRO": suministro,
                "PERIODO_ANIO_MES": periodo,
                "COMPENSACION_TOTAL_ORIGINAL": comp_total_raw,
                "COMPENSACION_TOTAL_NEGATIVA": comp_total_neg,
                "ESTADO": estado,
                "OBSERVACION": observacion,
                "FILAS_ARCHIVO": filas_archivo,
                "COINCIDENCIAS_VALOR": coincidencias,
                "RUTA_ARCHIVO_FINAL": ruta_archivo_final,
                "CAPTURA_ERROR": captura_error
            })

        if dataframes_ok:
            df_consolidado = pd.concat(dataframes_ok, ignore_index=True)
        else:
            df_consolidado = pd.DataFrame()

        df_consolidado.to_excel(RUTA_CONSOLIDADO, index=False)

        df_resultados = pd.DataFrame(resultados)

        resumen_general = pd.DataFrame({
            "METRICA": [
                "TOTAL_REGISTROS_BASE",
                "PROCESADOS_OK",
                "PROCESADOS_SIN_COINCIDENCIA_VALOR",
                "ERRORES",
                "ARCHIVOS_INDIVIDUALES_GENERADOS",
                "FILAS_CONSOLIDADO"
            ],
            "VALOR": [
                len(df_base),
                int((df_resultados["ESTADO"] == "OK").sum()),
                int((df_resultados["ESTADO"] == "SIN_COINCIDENCIA_VALOR").sum()),
                int((df_resultados["ESTADO"] == "ERROR").sum()),
                int(df_resultados["RUTA_ARCHIVO_FINAL"].fillna("").ne("").sum()),
                len(df_consolidado)
            ]
        })

        with pd.ExcelWriter(RUTA_RESUMEN, engine="openpyxl") as writer:
            df_resultados.to_excel(writer, sheet_name="DETALLE_PROCESO", index=False)
            resumen_general.to_excel(writer, sheet_name="RESUMEN", index=False)

        print("\n" + "=" * 80)
        print("PROCESO FINALIZADO")
        print("=" * 80)
        print(f"Ruta output: {RUTA_OUTPUT}")
        print(f"Archivos individuales: {RUTA_INDIVIDUALES}")
        print(f"Consolidado: {RUTA_CONSOLIDADO}")
        print(f"Resumen: {RUTA_RESUMEN}")
        print(f"Errores detalle: {RUTA_LOG_ERRORES}")
        print(f"Capturas: {RUTA_CAPTURAS}")
        print("-" * 80)
        print(f"Total base: {len(df_base)}")
        print(f"OK: {(df_resultados['ESTADO'] == 'OK').sum()}")
        print(f"Sin coincidencia valor: {(df_resultados['ESTADO'] == 'SIN_COINCIDENCIA_VALOR').sum()}")
        print(f"Errores: {(df_resultados['ESTADO'] == 'ERROR').sum()}")

    finally:
        driver.quit()


# =========================================================
# EJECUCIÓN
# =========================================================

if __name__ == "__main__":
    main()

[1/73] Procesando suministro: 3192402
[2/73] Procesando suministro: 3119706
[3/73] Procesando suministro: 3114112
[4/73] Procesando suministro: 3116481
[5/73] Procesando suministro: 3111251
[6/73] Procesando suministro: 3149697
[7/73] Procesando suministro: 3118930
[8/73] Procesando suministro: 614300
[9/73] Procesando suministro: 934168
[10/73] Procesando suministro: 935585
[11/73] Procesando suministro: 1637364
[12/73] Procesando suministro: 1866272
[13/73] Procesando suministro: 3035653
[14/73] Procesando suministro: 3065158
[15/73] Procesando suministro: 3115049
[16/73] Procesando suministro: 2549681
[17/73] Procesando suministro: 1611710
[18/73] Procesando suministro: 3136179
[19/73] Procesando suministro: 3137500
[20/73] Procesando suministro: 3137530
[21/73] Procesando suministro: 22310
[22/73] Procesando suministro: 3134165
[23/73] Procesando suministro: 3149697
[24/73] Procesando suministro: 3104780
[25/73] Procesando suministro: 2308403
[26/73] Procesando suministro: 22310
[2

In [4]:
# =========================================================
# AUTOMATIZACIÓN NASCA ENEL - DESCARGA Y CONSOLIDACIÓN
# VERSIÓN AJUSTADA PARA:
# - consulta facturación en nueva ventana
# - múltiples ventanas abiertas
# - tab que no siempre hace click
# =========================================================

import os
import re
import time
import shutil
import traceback
from pathlib import Path

import pandas as pd
import numpy as np

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException


# =========================================================
# CONFIGURACIÓN
# =========================================================

URL_LOGIN = "https://nasca.enel.com/app-web-synergia/#/consulta-cuenta"

USUARIO = "pe15733437"
CONTRASENA = "Cds1987200523@"   # <-- CAMBIAR

RUTA_EXCEL_BASE = r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\BUSCADOR DE COMPENSACIONES OFICIO\IMPUT\SUMINISTROS.xlsx"

RUTA_OUTPUT = r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\BUSCADOR DE COMPENSACIONES OFICIO\OUT-PUT"

RUTA_DESCARGAS = os.path.join(RUTA_OUTPUT, "DESCARGAS_TEMP")
RUTA_INDIVIDUALES = os.path.join(RUTA_OUTPUT, "EXCEL_INDIVIDUALES")
RUTA_LOGS = os.path.join(RUTA_OUTPUT, "LOGS")
RUTA_CAPTURAS = os.path.join(RUTA_OUTPUT, "CAPTURAS")

RUTA_CONSOLIDADO = os.path.join(RUTA_OUTPUT, "CONSOLIDADO_COMPENSACIONES.xlsx")
RUTA_RESUMEN = os.path.join(RUTA_OUTPUT, "RESUMEN_PROCESO.xlsx")
RUTA_LOG_ERRORES = os.path.join(RUTA_LOGS, "errores_detalle.txt")

TIMEOUT = 25
TIEMPO_DESCARGA_MAX = 60
MODO_HEADLESS = False

CHROMEDRIVER_PATH = None
# CHROMEDRIVER_PATH = r"C:\ruta\chromedriver.exe"


# =========================================================
# CREAR CARPETAS
# =========================================================

for ruta in [RUTA_OUTPUT, RUTA_DESCARGAS, RUTA_INDIVIDUALES, RUTA_LOGS, RUTA_CAPTURAS]:
    os.makedirs(ruta, exist_ok=True)


# =========================================================
# UTILITARIOS
# =========================================================

def limpiar_nombre_archivo(texto: str) -> str:
    texto = str(texto).strip()
    texto = re.sub(r'[<>:"/\\|?*]+', '_', texto)
    texto = re.sub(r'\s+', '_', texto)
    return texto


def convertir_numero(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip()
    if valor == "":
        return np.nan

    valor = valor.replace(" ", "")

    if "." in valor and "," in valor:
        valor = valor.replace(".", "").replace(",", ".")
    else:
        if "," in valor:
            valor = valor.replace(",", ".")

    try:
        return float(valor)
    except:
        return np.nan


def esperar_archivo_descargado(carpeta_descargas, timeout=60, prefijo="Reporte"):
    inicio = time.time()

    while time.time() - inicio < timeout:
        archivos = list(Path(carpeta_descargas).glob(f"{prefijo}*"))

        archivos_validos = [
            a for a in archivos
            if not str(a).endswith(".crdownload") and not str(a).endswith(".tmp")
        ]

        if archivos_validos:
            archivo_reciente = max(archivos_validos, key=lambda x: x.stat().st_mtime)

            size_1 = archivo_reciente.stat().st_size
            time.sleep(1.5)
            size_2 = archivo_reciente.stat().st_size

            if size_1 == size_2 and size_2 > 0:
                return str(archivo_reciente)

        time.sleep(1)

    return None


def eliminar_reportes_previos(carpeta_descargas, prefijo="Reporte"):
    archivos = list(Path(carpeta_descargas).glob(f"{prefijo}*"))
    for archivo in archivos:
        try:
            os.remove(archivo)
        except:
            pass


def iniciar_driver():
    chrome_options = Options()

    prefs = {
        "download.default_directory": RUTA_DESCARGAS,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
        "plugins.always_open_pdf_externally": True,
    }
    chrome_options.add_experimental_option("prefs", prefs)

    if MODO_HEADLESS:
        chrome_options.add_argument("--headless=new")

    chrome_options.add_argument("--start-maximized")
    chrome_options.add_argument("--disable-notifications")
    chrome_options.add_argument("--disable-popup-blocking")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    if CHROMEDRIVER_PATH:
        service = Service(CHROMEDRIVER_PATH)
        driver = webdriver.Chrome(service=service, options=chrome_options)
    else:
        driver = webdriver.Chrome(options=chrome_options)

    return driver


def leer_excel_descargado(ruta_excel):
    ext = Path(ruta_excel).suffix.lower()

    if ext == ".xlsx":
        return pd.read_excel(ruta_excel, engine="openpyxl")
    elif ext == ".xls":
        try:
            return pd.read_excel(ruta_excel)
        except:
            return pd.read_excel(ruta_excel, engine="xlrd")
    else:
        return pd.read_excel(ruta_excel)


def guardar_captura(driver, nombre_base):
    ruta = os.path.join(RUTA_CAPTURAS, f"{limpiar_nombre_archivo(nombre_base)}.png")
    try:
        driver.save_screenshot(ruta)
        return ruta
    except:
        return ""


def cerrar_ventanas_secundarias(driver):
    """
    Cierra todas las ventanas excepto la primera.
    """
    try:
        handles = driver.window_handles
        if not handles:
            return

        principal = handles[0]

        for h in handles[1:]:
            try:
                driver.switch_to.window(h)
                driver.close()
            except:
                pass

        driver.switch_to.window(principal)
        time.sleep(1)
    except:
        pass


def esperar_documento_listo(driver, timeout=20):
    WebDriverWait(driver, timeout).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )


def click_js(driver, element):
    driver.execute_script("arguments[0].click();", element)


def agregar_columnas_y_guardar(
    ruta_archivo,
    interrupcion,
    suministro,
    periodo_anio_mes,
    compensacion_total,
    ruta_guardado_final
):
    df = leer_excel_descargado(ruta_archivo)
    df.columns = [str(c).strip() for c in df.columns]

    df["INTERRUPCION"] = interrupcion
    df["SUMINISTRO"] = str(suministro)
    df["PERIODO_ANIO_MES"] = str(periodo_anio_mes)

    comp = convertir_numero(compensacion_total)
    if pd.notna(comp):
        comp = -abs(comp)

    df["COMPENSACION_TOTAL"] = comp

    if "Valor" in df.columns:
        df["VALOR_NUM"] = df["Valor"].apply(convertir_numero)
    else:
        df["VALOR_NUM"] = np.nan

    if pd.notna(comp):
        df["COINCIDE_COMPENSACION"] = np.isclose(
            df["VALOR_NUM"].round(2),
            round(comp, 2),
            atol=0.01
        )
    else:
        df["COINCIDE_COMPENSACION"] = False

    df.to_excel(ruta_guardado_final, index=False)
    return df


# =========================================================
# LOGIN Y NAVEGACIÓN
# =========================================================

def hacer_login(driver):
    driver.get(URL_LOGIN)

    usuario_xpath = "/html/body/app-root/app-login/div/div/div/div[2]/div[1]/form/div[1]/span/input"
    password_xpath = "/html/body/app-root/app-login/div/div/div/div[2]/div[1]/form/div[2]/span/input"

    usuario = WebDriverWait(driver, TIMEOUT).until(
        EC.visibility_of_element_located((By.XPATH, usuario_xpath))
    )
    usuario.click()
    usuario.send_keys(Keys.CONTROL, "a")
    usuario.send_keys(Keys.DELETE)
    usuario.send_keys(USUARIO)

    password = WebDriverWait(driver, TIMEOUT).until(
        EC.visibility_of_element_located((By.XPATH, password_xpath))
    )
    password.click()
    password.send_keys(Keys.CONTROL, "a")
    password.send_keys(Keys.DELETE)
    password.send_keys(CONTRASENA)
    password.send_keys(Keys.ENTER)

    WebDriverWait(driver, TIMEOUT).until(
        EC.presence_of_element_located((By.XPATH, "/html/body/app-root/app-layout"))
    )

    esperar_documento_listo(driver)
    time.sleep(2)


def abrir_menu_consulta_cuenta(driver):
    click_targets = [
        "/html/body/app-root/app-layout/div/app-topbar/div/button",
        "/html/body/app-root/app-layout/div/app-topbar/div/button/i"
    ]

    ok = False
    for xp in click_targets:
        try:
            boton = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, xp))
            )
            try:
                boton.click()
            except:
                click_js(driver, boton)
            ok = True
            break
        except:
            pass

    if not ok:
        raise TimeoutException("No se pudo abrir el menú lateral.")

    time.sleep(1.5)

    menu_1 = "/html/body/app-root/app-layout/div/div[1]/app-sidebar/app-menu/ul/li[1]/ul/li/a/span[1]"
    sub_menu = "/html/body/app-root/app-layout/div/div[1]/app-sidebar/app-menu/ul/li[1]/ul/li/ul/li/a/span[1]"

    elem1 = WebDriverWait(driver, TIMEOUT).until(
        EC.element_to_be_clickable((By.XPATH, menu_1))
    )
    try:
        elem1.click()
    except:
        click_js(driver, elem1)

    time.sleep(1)

    elem2 = WebDriverWait(driver, TIMEOUT).until(
        EC.element_to_be_clickable((By.XPATH, sub_menu))
    )
    try:
        elem2.click()
    except:
        click_js(driver, elem2)

    time.sleep(4)
    esperar_documento_listo(driver)


def volver_a_consulta_cuenta(driver):
    cerrar_ventanas_secundarias(driver)
    driver.get("https://nasca.enel.com/app-web-synergia/#/consulta-cuenta")
    time.sleep(4)

    WebDriverWait(driver, TIMEOUT).until(
        EC.presence_of_element_located((By.XPATH, "/html/body/app-root/app-consulta-cuenta"))
    )
    esperar_documento_listo(driver)
    time.sleep(2)


# =========================================================
# BÚSQUEDA DE SUMINISTRO
# =========================================================

def buscar_suministro_en_consulta_cuenta(driver, suministro):
    posibles_xpaths_input = [
        "/html/body/app-root/app-consulta-cuenta/div/div/div/div/div/div/div[3]/div/div/div[1]/div/div[2]/input",
        "/html/body/app-root/app-consulta-cuenta/div[2]//input",
        "//app-consulta-cuenta//input"
    ]

    campo = None
    for xp in posibles_xpaths_input:
        try:
            campo = WebDriverWait(driver, 12).until(
                EC.visibility_of_element_located((By.XPATH, xp))
            )
            if campo:
                break
        except:
            pass

    if campo is None:
        raise TimeoutException("No se encontró el campo de suministro.")

    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", campo)
    time.sleep(1)

    try:
        campo.click()
    except:
        click_js(driver, campo)

    campo.send_keys(Keys.CONTROL, "a")
    campo.send_keys(Keys.DELETE)
    campo.send_keys(str(suministro))
    time.sleep(1)

    try:
        campo.send_keys(Keys.ENTER)
    except:
        pass

    time.sleep(3)

    posibles_xpaths_boton = [
        "/html/body/app-root/app-consulta-cuenta/div[2]/div/div/div/div/div/div[3]/div/div/div[2]/div/div[1]/p-button/button",
        "/html/body/app-root/app-consulta-cuenta/div[2]/div/div/div/div/div/div[3]/div/div/div[2]/div/div[1]/p-button/button/span[2]",
        "//app-consulta-cuenta//p-button//button"
    ]

    for xp in posibles_xpaths_boton:
        try:
            boton = WebDriverWait(driver, 6).until(
                EC.element_to_be_clickable((By.XPATH, xp))
            )
            try:
                boton.click()
            except:
                click_js(driver, boton)
            break
        except:
            pass

    time.sleep(4)
    esperar_documento_listo(driver, timeout=12)


def validar_resultado_busqueda(driver):
    posibles = [
        "//app-consulta-cuenta//button[.//img]",
        "/html/body/app-root/app-consulta-cuenta//button[2]"
    ]
    for xp in posibles:
        elems = driver.find_elements(By.XPATH, xp)
        if len(elems) > 0:
            return True
    return False


# =========================================================
# FACTURACIÓN EN NUEVA VENTANA
# =========================================================

def abrir_consulta_facturacion(driver, suministro=None):
    """
    Cierra ventanas viejas, hace click, detecta la nueva ventana real
    y cambia a ella.
    """
    cerrar_ventanas_secundarias(driver)

    ventana_original = driver.current_window_handle
    ventanas_antes = set(driver.window_handles)

    posibles_xpaths_fact = [
        "/html/body/app-root/app-consulta-cuenta/div[2]/div/div/div/div/div/div[1]/div/button[2]",
        "/html/body/app-root/app-consulta-cuenta/div[2]/div/div/div/div/div/div[1]/div/button[2]/img",
        "//app-consulta-cuenta//button[.//img]"
    ]

    boton_encontrado = False
    ultimo_error = None

    for xp in posibles_xpaths_fact:
        try:
            boton = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, xp))
            )
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", boton)
            time.sleep(1)

            try:
                WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((By.XPATH, xp))
                )
                try:
                    boton.click()
                except:
                    click_js(driver, boton)
            except:
                click_js(driver, boton)

            boton_encontrado = True
            break
        except Exception as e:
            ultimo_error = e

    if not boton_encontrado:
        raise TimeoutException(f"No se encontró el botón de consulta facturación. {ultimo_error}")

    # Esperar nueva ventana
    WebDriverWait(driver, 15).until(
        lambda d: len(d.window_handles) > len(ventanas_antes)
    )

    ventanas_despues = driver.window_handles
    nuevas_ventanas = [h for h in ventanas_despues if h not in ventanas_antes]

    if not nuevas_ventanas:
        raise TimeoutException("Se hizo click, pero no apareció una nueva ventana.")

    # Normalmente la correcta es la última creada
    nueva_ventana = nuevas_ventanas[-1]
    driver.switch_to.window(nueva_ventana)

    # No exigir URL exacta; validar por URL O por presencia de elementos de facturación
    try:
        WebDriverWait(driver, 20).until(
            lambda d: (
                "facturacion" in d.current_url.lower()
                or len(d.find_elements(By.XPATH, '//*[@id="p-tabpanel-2-label"]')) > 0
                or len(d.find_elements(By.XPATH, '//*[@id="pr_id_3-table"]')) > 0
            )
        )
    except:
        nombre = f"error_facturacion_{suministro}" if suministro else "error_facturacion"
        ruta = guardar_captura(driver, nombre)
        raise TimeoutException(f"No cargó consulta-facturacion luego del click. Captura: {ruta}")

    esperar_documento_listo(driver)
    time.sleep(3)

    return ventana_original


def cerrar_ventana_facturacion_y_volver(driver, ventana_original):
    try:
        actual = driver.current_window_handle
        if actual != ventana_original:
            driver.close()
    except:
        pass

    driver.switch_to.window(ventana_original)
    time.sleep(2)


# =========================================================
# DESCARGA EN FACTURACIÓN
# =========================================================

def intentar_click_tab_facturacion(driver):
    """
    A veces el tab no se deja clickear, o ya está activo.
    """
    posibles_tabs = [
        '//*[@id="p-tabpanel-2-label"]/span[1]',
        '//*[@id="p-tabpanel-2-label"]',
        '//button[@id="p-tabpanel-2-label"]',
        '//*[contains(@id,"p-tabpanel") and contains(@id,"label")]'
    ]

    for xp in posibles_tabs:
        try:
            elem = WebDriverWait(driver, 6).until(
                EC.presence_of_element_located((By.XPATH, xp))
            )
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", elem)
            time.sleep(0.5)

            try:
                elem.click()
            except:
                click_js(driver, elem)

            time.sleep(2)
            return True
        except:
            pass

    return False


def descargar_excel_facturacion(driver):
    """
    Intenta:
    1) click al tab
    2) click directo a descarga
    3) JS click si el click normal falla
    """

    # Intentar activar tab, pero no es obligatorio para continuar
    intentar_click_tab_facturacion(driver)

    posibles_descarga = [
        '//*[@id="pr_id_3-table"]/tfoot/tr/td[2]/label[1]',
        '//*[@id="pr_id_3-table"]//tfoot//label[1]',
        '//table[@id="pr_id_3-table"]//tfoot//label',
        '//*[contains(@id,"pr_id_3-table")]//tfoot//label'
    ]

    ultimo_error = None

    for xp in posibles_descarga:
        try:
            btn = WebDriverWait(driver, 12).until(
                EC.presence_of_element_located((By.XPATH, xp))
            )

            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
            time.sleep(1)

            try:
                btn.click()
            except:
                click_js(driver, btn)

            time.sleep(3)
            return
        except Exception as e:
            ultimo_error = e

    ruta = guardar_captura(driver, "error_descarga_facturacion")
    raise TimeoutException(f"No se pudo hacer click en el botón de descarga. Captura: {ruta}. Detalle: {ultimo_error}")


# =========================================================
# PROCESAMIENTO PRINCIPAL
# =========================================================

def main():
    df_base = pd.read_excel(RUTA_EXCEL_BASE, dtype=str)
    df_base.columns = [str(c).strip().upper() for c in df_base.columns]

    columnas_requeridas = ["INTERRUPCION", "SUMINISTRO", "PERIODO_ANIO_MES", "COMPENSACION_TOTAL"]
    faltantes = [c for c in columnas_requeridas if c not in df_base.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas en el Excel base: {faltantes}")

    df_base = df_base[columnas_requeridas].copy()
    df_base["COMPENSACION_TOTAL_NUM"] = df_base["COMPENSACION_TOTAL"].apply(convertir_numero)
    df_base["COMPENSACION_TOTAL_NEG"] = -df_base["COMPENSACION_TOTAL_NUM"].abs()

    # PRUEBA
    # df_base = df_base.head(1)

    resultados = []
    dataframes_ok = []

    if os.path.exists(RUTA_LOG_ERRORES):
        os.remove(RUTA_LOG_ERRORES)

    driver = iniciar_driver()

    try:
        hacer_login(driver)
        abrir_menu_consulta_cuenta(driver)

        total = len(df_base)

        for i, row in df_base.iterrows():
            interrupcion = str(row["INTERRUPCION"]).strip()
            suministro = str(row["SUMINISTRO"]).strip()
            periodo = str(row["PERIODO_ANIO_MES"]).strip()
            comp_total_raw = row["COMPENSACION_TOTAL"]
            comp_total_neg = row["COMPENSACION_TOTAL_NEG"]

            nombre_archivo = limpiar_nombre_archivo(f"{interrupcion}_{periodo}_{suministro}.xlsx")
            ruta_final_individual = os.path.join(RUTA_INDIVIDUALES, nombre_archivo)

            estado = "OK"
            observacion = ""
            ruta_archivo_final = ""
            filas_archivo = 0
            coincidencias = 0
            ruta_screenshot = ""

            print(f"[{i+1}/{total}] Procesando suministro: {suministro}")

            try:
                cerrar_ventanas_secundarias(driver)
                eliminar_reportes_previos(RUTA_DESCARGAS, prefijo="Reporte")

                buscar_suministro_en_consulta_cuenta(driver, suministro)

                if not validar_resultado_busqueda(driver):
                    raise Exception(f"No hubo resultado visible para el suministro {suministro}")

                ventana_original = abrir_consulta_facturacion(driver, suministro=suministro)

                descargar_excel_facturacion(driver)

                ruta_descargada = esperar_archivo_descargado(
                    RUTA_DESCARGAS,
                    timeout=TIEMPO_DESCARGA_MAX,
                    prefijo="Reporte"
                )

                if not ruta_descargada:
                    raise FileNotFoundError("No se detectó la descarga del archivo Reporte.")

                cerrar_ventana_facturacion_y_volver(driver, ventana_original)

                if os.path.exists(ruta_final_individual):
                    os.remove(ruta_final_individual)

                shutil.move(ruta_descargada, ruta_final_individual)

                df_ind = agregar_columnas_y_guardar(
                    ruta_archivo=ruta_final_individual,
                    interrupcion=interrupcion,
                    suministro=suministro,
                    periodo_anio_mes=periodo,
                    compensacion_total=comp_total_raw,
                    ruta_guardado_final=ruta_final_individual
                )

                filas_archivo = len(df_ind)
                coincidencias = int(df_ind["COINCIDE_COMPENSACION"].sum()) if "COINCIDE_COMPENSACION" in df_ind.columns else 0
                ruta_archivo_final = ruta_final_individual

                dataframes_ok.append(df_ind)

                if coincidencias == 0:
                    estado = "SIN_COINCIDENCIA_VALOR"
                    observacion = "Se descargó archivo, pero ninguna fila de 'Valor' coincide con COMPENSACION_TOTAL negativa."

            except Exception as e:
                estado = "ERROR"
                observacion = str(e)

                ruta_screenshot = guardar_captura(driver, f"error_{suministro}_{interrupcion}")

                with open(RUTA_LOG_ERRORES, "a", encoding="utf-8") as f:
                    f.write("=" * 120 + "\n")
                    f.write(f"SUMINISTRO: {suministro}\n")
                    f.write(f"INTERRUPCION: {interrupcion}\n")
                    f.write(f"PERIODO_ANIO_MES: {periodo}\n")
                    f.write(f"ERROR: {str(e)}\n")
                    if ruta_screenshot:
                        f.write(f"CAPTURA: {ruta_screenshot}\n")
                    f.write(traceback.format_exc())
                    f.write("\n")

                try:
                    cerrar_ventanas_secundarias(driver)
                except:
                    pass

            finally:
                resultados.append({
                    "INTERRUPCION": interrupcion,
                    "SUMINISTRO": suministro,
                    "PERIODO_ANIO_MES": periodo,
                    "COMPENSACION_TOTAL_ORIGINAL": comp_total_raw,
                    "COMPENSACION_TOTAL_NEGATIVA": comp_total_neg,
                    "ESTADO": estado,
                    "OBSERVACION": observacion,
                    "FILAS_ARCHIVO": filas_archivo,
                    "COINCIDENCIAS_VALOR": coincidencias,
                    "RUTA_ARCHIVO_FINAL": ruta_archivo_final,
                    "RUTA_SCREENSHOT": ruta_screenshot
                })

                try:
                    volver_a_consulta_cuenta(driver)
                except:
                    try:
                        cerrar_ventanas_secundarias(driver)
                        driver.get("https://nasca.enel.com/app-web-synergia/#/consulta-cuenta")
                        time.sleep(5)
                    except:
                        pass

        if dataframes_ok:
            df_consolidado = pd.concat(dataframes_ok, ignore_index=True)
        else:
            df_consolidado = pd.DataFrame()

        df_consolidado.to_excel(RUTA_CONSOLIDADO, index=False)

        df_resultados = pd.DataFrame(resultados)

        resumen_general = pd.DataFrame({
            "METRICA": [
                "TOTAL_REGISTROS_BASE",
                "PROCESADOS_OK",
                "PROCESADOS_SIN_COINCIDENCIA_VALOR",
                "ERRORES",
                "ARCHIVOS_INDIVIDUALES_GENERADOS",
                "FILAS_CONSOLIDADO"
            ],
            "VALOR": [
                len(df_base),
                int((df_resultados["ESTADO"] == "OK").sum()),
                int((df_resultados["ESTADO"] == "SIN_COINCIDENCIA_VALOR").sum()),
                int((df_resultados["ESTADO"] == "ERROR").sum()),
                int(df_resultados["RUTA_ARCHIVO_FINAL"].fillna("").ne("").sum()),
                len(df_consolidado)
            ]
        })

        with pd.ExcelWriter(RUTA_RESUMEN, engine="openpyxl") as writer:
            df_resultados.to_excel(writer, sheet_name="DETALLE_PROCESO", index=False)
            resumen_general.to_excel(writer, sheet_name="RESUMEN", index=False)

        print("\n" + "=" * 80)
        print("PROCESO FINALIZADO")
        print("=" * 80)
        print(f"Ruta output: {RUTA_OUTPUT}")
        print(f"Consolidado: {RUTA_CONSOLIDADO}")
        print(f"Resumen: {RUTA_RESUMEN}")
        print(f"Log errores: {RUTA_LOG_ERRORES}")
        print(f"Capturas: {RUTA_CAPTURAS}")
        print("-" * 80)
        print(f"Total base: {len(df_base)}")
        print(f"OK: {(df_resultados['ESTADO'] == 'OK').sum()}")
        print(f"Sin coincidencia valor: {(df_resultados['ESTADO'] == 'SIN_COINCIDENCIA_VALOR').sum()}")
        print(f"Errores: {(df_resultados['ESTADO'] == 'ERROR').sum()}")

    finally:
        driver.quit()


# =========================================================
# EJECUCIÓN
# =========================================================

if __name__ == "__main__":
    main()

[1/73] Procesando suministro: 3192402
[2/73] Procesando suministro: 3119706


c:\Users\P760033411\AppData\Local\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


[3/73] Procesando suministro: 3114112


c:\Users\P760033411\AppData\Local\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


[4/73] Procesando suministro: 3116481


c:\Users\P760033411\AppData\Local\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


... (135 outputs left out)

c:\Users\P760033411\AppData\Local\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


[73/73] Procesando suministro: 3134376


c:\Users\P760033411\AppData\Local\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\P760033411\AppData\Local\Temp\ipykernel_18176\2550488905.py:764: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_consolidado = pd.concat(dataframes_ok, ignore_index=True)



PROCESO FINALIZADO
Ruta output: C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\BUSCADOR DE COMPENSACIONES OFICIO\OUT-PUT
Consolidado: C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\BUSCADOR DE COMPENSACIONES OFICIO\OUT-PUT\CONSOLIDADO_COMPENSACIONES.xlsx
Resumen: C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\BUSCADOR DE COMPENSACIONES OFICIO\OUT-PUT\RESUMEN_PROCESO.xlsx
Log errores: C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\BUSCADOR DE COMPENSACIONES OFICIO\OUT-PUT\LOGS\errores_detalle.txt
Capturas: C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\BUSCADOR DE COMPENSACIONES OFICIO\OUT-PUT\CAPTURAS
--------------------------------------------------------------------------------
Total base: 73
OK: 28
Sin coincidencia valor: 44
Errores: 1


In [1]:
import os
import re
import unicodedata
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter


# =========================================================
# CONFIGURACIÓN
# =========================================================

RUTA_EXCELS_INDIVIDUALES = r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\BUSCADOR DE COMPENSACIONES OFICIO\OUT-PUT\EXCEL_INDIVIDUALES"

RUTA_SUMINISTROS = r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\BUSCADOR DE COMPENSACIONES OFICIO\IMPUT\SUMINISTROS.xlsx"

RUTA_OUTPUT = r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\BUSCADOR DE COMPENSACIONES OFICIO\OUT-PUT"

ARCHIVO_SALIDA_CONSOLIDADO = os.path.join(RUTA_OUTPUT, "01_CONSOLIDADO_EXCEL_INDIVIDUALES.xlsx")
ARCHIVO_SALIDA_CRUCE = os.path.join(RUTA_OUTPUT, "02_CRUCE_COINCIDENCIAS.xlsx")

COMPARAR_USANDO_ABS = True


# =========================================================
# FUNCIONES AUXILIARES
# =========================================================

def normalizar_texto(txt):
    if pd.isna(txt):
        return ""
    txt = str(txt).strip().lower()
    txt = unicodedata.normalize("NFKD", txt).encode("ascii", "ignore").decode("ascii")
    txt = re.sub(r"[^a-z0-9]+", "", txt)
    return txt


def convertir_numero(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip()

    if valor == "":
        return np.nan

    valor = valor.replace(" ", "")

    if "." in valor and "," in valor:
        valor = valor.replace(".", "").replace(",", ".")
    else:
        if "," in valor:
            valor = valor.replace(",", ".")

    try:
        return float(valor)
    except:
        return np.nan


def convertir_fecha_mixta(fecha):
    if pd.isna(fecha):
        return pd.NaT

    if isinstance(fecha, (datetime, pd.Timestamp)):
        return pd.to_datetime(fecha)

    texto = str(fecha).strip().lower()

    if texto == "":
        return pd.NaT

    meses = {
        "ene": "01", "feb": "02", "mar": "03", "abr": "04",
        "may": "05", "jun": "06", "jul": "07", "ago": "08",
        "sep": "09", "oct": "10", "nov": "11", "dic": "12"
    }

    m = re.match(r"^(\d{1,2})-([a-z]{3})-(\d{4})$", texto)
    if m:
        dia, mes_txt, anio = m.groups()
        mes_num = meses.get(mes_txt)
        if mes_num:
            return pd.to_datetime(f"{anio}-{mes_num}-{int(dia):02d}", errors="coerce")

    return pd.to_datetime(texto, dayfirst=True, errors="coerce")


def fecha_a_yyyymm(fecha):
    if pd.isna(fecha):
        return np.nan
    return int(fecha.strftime("%Y%m"))


def extraer_datos_desde_nombre(nombre_archivo):
    base = Path(nombre_archivo).stem
    partes = base.split("_")

    interrupcion = None
    periodo = None
    suministro = None

    if len(partes) >= 3:
        interrupcion = partes[0]
        periodo = partes[1]
        suministro = partes[2]

    return interrupcion, periodo, suministro


def es_compensacion_4hrs(descripcion):
    txt = normalizar_texto(descripcion)
    return (
        ("compens" in txt or "compensacion" in txt)
        and "interrupcion" in txt
        and ("4hrs" in txt or "4hr" in txt or "4horas" in txt)
    )


def ajustar_ancho_y_estilo_excel(ruta_excel):
    wb = load_workbook(ruta_excel)

    fill_header = PatternFill("solid", fgColor="1F4E78")
    font_header = Font(color="FFFFFF", bold=True)
    align = Alignment(horizontal="center", vertical="center")

    for ws in wb.worksheets:
        ws.freeze_panes = "A2"

        if ws.max_row >= 1 and ws.max_column >= 1:
            ws.auto_filter.ref = ws.dimensions

        for cell in ws[1]:
            cell.fill = fill_header
            cell.font = font_header
            cell.alignment = align

        for col_idx in range(1, ws.max_column + 1):
            max_len = 0
            col_letter = get_column_letter(col_idx)

            for row in range(1, ws.max_row + 1):
                value = ws.cell(row=row, column=col_idx).value
                if value is not None:
                    max_len = max(max_len, len(str(value)))

            ws.column_dimensions[col_letter].width = min(max(max_len + 2, 12), 40)

    wb.save(ruta_excel)


def preparar_columnas(df):
    df.columns = [str(c).strip() for c in df.columns]
    return df


# =========================================================
# 1) CONSOLIDAR EXCELES INDIVIDUALES
# =========================================================

def consolidar_excels_individuales():
    archivos = sorted(Path(RUTA_EXCELS_INDIVIDUALES).glob("*.xlsx"))

    if not archivos:
        raise FileNotFoundError(f"No se encontraron archivos .xlsx en: {RUTA_EXCELS_INDIVIDUALES}")

    lista_todos = []

    for archivo in archivos:
        try:
            df = pd.read_excel(archivo, engine="openpyxl")
            df = preparar_columnas(df)

            interrupcion_arch, periodo_arch, suministro_arch = extraer_datos_desde_nombre(archivo.name)

            if "INTERRUPCION" not in df.columns:
                df["INTERRUPCION"] = interrupcion_arch

            if "PERIODO_ANIO_MES" not in df.columns:
                df["PERIODO_ANIO_MES"] = periodo_arch

            if "SUMINISTRO" not in df.columns:
                df["SUMINISTRO"] = suministro_arch

            df["INTERRUPCION"] = df["INTERRUPCION"].fillna(interrupcion_arch).astype(str)
            df["PERIODO_ANIO_MES"] = df["PERIODO_ANIO_MES"].fillna(periodo_arch).astype(str)
            df["SUMINISTRO"] = df["SUMINISTRO"].fillna(suministro_arch).astype(str)

            df["ARCHIVO_ORIGEN"] = archivo.name

            # =====================================================
            # MARCA SI EL EXCEL TIENE DATOS APARTE DEL ENCABEZADO
            # =====================================================
            tiene_datos_excel = len(df) > 0

            if "Valor" in df.columns:
                df["VALOR_NUM"] = df["Valor"].apply(convertir_numero)
            else:
                df["VALOR_NUM"] = np.nan

            df["ESTADO_EXCEL"] = "CON_DATOS" if tiene_datos_excel else "SIN_DATOS"

            if "Fecha Facturación" in df.columns:
                df["FECHA_FACTURACION_DT"] = df["Fecha Facturación"].apply(convertir_fecha_mixta)
                df["FECHA_FACTURACION_YYYYMM"] = df["FECHA_FACTURACION_DT"].apply(fecha_a_yyyymm)
            else:
                df["FECHA_FACTURACION_DT"] = pd.NaT
                df["FECHA_FACTURACION_YYYYMM"] = np.nan

            if "Descripción del Cargo" in df.columns:
                df["ES_COMPENSACION_4HRS"] = df["Descripción del Cargo"].apply(es_compensacion_4hrs)
            else:
                df["ES_COMPENSACION_4HRS"] = False

            lista_todos.append(df)

        except Exception as e:
            print(f"Error leyendo {archivo.name}: {e}")

    if not lista_todos:
        raise ValueError("No se pudo leer ningún Excel individual.")

    df_todos = pd.concat(lista_todos, ignore_index=True, sort=False)
    df_4hrs = df_todos[df_todos["ES_COMPENSACION_4HRS"] == True].copy()

    with pd.ExcelWriter(ARCHIVO_SALIDA_CONSOLIDADO, engine="openpyxl") as writer:
        df_todos.to_excel(writer, sheet_name="Todos_Consolidados", index=False)
        df_4hrs.to_excel(writer, sheet_name="Solo_Compens_4Hrs", index=False)

    ajustar_ancho_y_estilo_excel(ARCHIVO_SALIDA_CONSOLIDADO)

    print("=" * 80)
    print("CONSOLIDADO GENERADO")
    print("=" * 80)
    print(f"Archivo: {ARCHIVO_SALIDA_CONSOLIDADO}")
    print(f"Filas hoja Todos_Consolidados: {len(df_todos)}")
    print(f"Filas hoja Solo_Compens_4Hrs: {len(df_4hrs)}")

    return df_todos


# =========================================================
# 2) CRUCE DE COINCIDENCIAS
# =========================================================

def generar_cruce_coincidencias(df_todos):
    df_base = pd.read_excel(RUTA_SUMINISTROS, dtype=str, engine="openpyxl")
    df_base.columns = [str(c).strip().upper() for c in df_base.columns]

    columnas_req = ["INTERRUPCION", "SUMINISTRO", "PERIODO_ANIO_MES", "COMPENSACION_TOTAL"]
    faltantes = [c for c in columnas_req if c not in df_base.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas en SUMINISTROS.xlsx: {faltantes}")

    df_base = df_base[columnas_req].copy()

    # =====================================================
    # NORMALIZAR BASE
    # =====================================================
    df_base["INTERRUPCION_BASE"] = df_base["INTERRUPCION"].astype(str).str.strip()
    df_base["SUMINISTRO_BASE"] = df_base["SUMINISTRO"].astype(str).str.strip()
    df_base["PERIODO_ANIO_MES_BASE"] = pd.to_numeric(df_base["PERIODO_ANIO_MES"], errors="coerce")
    df_base["COMPENSACION_TOTAL_BASE"] = df_base["COMPENSACION_TOTAL"]
    df_base["COMPENSACION_TOTAL_BASE_NUM"] = df_base["COMPENSACION_TOTAL_BASE"].apply(convertir_numero)

    df_base_merge = df_base[[
        "INTERRUPCION_BASE",
        "SUMINISTRO_BASE",
        "PERIODO_ANIO_MES_BASE",
        "COMPENSACION_TOTAL_BASE",
        "COMPENSACION_TOTAL_BASE_NUM"
    ]].copy()

    # =====================================================
    # NORMALIZAR COMPILADO
    # =====================================================
    df_comp = df_todos.copy()

    df_comp["SUMINISTRO_COMP"] = df_comp["SUMINISTRO"].astype(str).str.strip()
    df_comp["INTERRUPCION_COMP"] = df_comp["INTERRUPCION"].astype(str).str.strip()
    df_comp["PERIODO_ANIO_MES_COMP"] = pd.to_numeric(df_comp["PERIODO_ANIO_MES"], errors="coerce")

    if "VALOR_NUM" not in df_comp.columns:
        df_comp["VALOR_NUM"] = np.nan

    if "FECHA_FACTURACION_YYYYMM" not in df_comp.columns:
        df_comp["FECHA_FACTURACION_YYYYMM"] = np.nan

    # =====================================================
    # MERGE POR SUMINISTRO
    # =====================================================
    df_merge = df_comp.merge(
        df_base_merge,
        left_on="SUMINISTRO_COMP",
        right_on="SUMINISTRO_BASE",
        how="inner"
    )

    # =====================================================
    # REGLAS DE COINCIDENCIA
    # =====================================================
    df_merge["CUMPLE_SUMINISTRO"] = df_merge["SUMINISTRO_COMP"] == df_merge["SUMINISTRO_BASE"]

    df_merge["CUMPLE_FECHA"] = (
        pd.to_numeric(df_merge["FECHA_FACTURACION_YYYYMM"], errors="coerce")
        <=
        pd.to_numeric(df_merge["PERIODO_ANIO_MES_BASE"], errors="coerce")
    )

    if COMPARAR_USANDO_ABS:
        df_merge["VALOR_COMPARE"] = df_merge["VALOR_NUM"].abs()
        df_merge["COMP_COMPARE"] = df_merge["COMPENSACION_TOTAL_BASE_NUM"].abs()
    else:
        df_merge["VALOR_COMPARE"] = df_merge["VALOR_NUM"]
        df_merge["COMP_COMPARE"] = df_merge["COMPENSACION_TOTAL_BASE_NUM"]

    df_merge["CUMPLE_VALOR"] = df_merge["VALOR_COMPARE"] >= df_merge["COMP_COMPARE"]

    df_merge["COINCIDE_FINAL"] = (
        df_merge["CUMPLE_SUMINISTRO"]
        & df_merge["CUMPLE_FECHA"]
        & df_merge["CUMPLE_VALOR"]
    )

    df_coincidencias = df_merge[df_merge["COINCIDE_FINAL"] == True].copy()

    # =====================================================
    # BASE SIN COINCIDENCIA
    # =====================================================
    base_cols = [
        "INTERRUPCION_BASE",
        "SUMINISTRO_BASE",
        "PERIODO_ANIO_MES_BASE",
        "COMPENSACION_TOTAL_BASE"
    ]

    if len(df_coincidencias) > 0:
        claves_con_match = df_coincidencias[base_cols].drop_duplicates()

        df_base_comparable = df_base_merge[base_cols].drop_duplicates()

        df_base_sin = df_base_comparable.merge(
            claves_con_match,
            on=base_cols,
            how="left",
            indicator=True
        )

        df_base_sin = df_base_sin[df_base_sin["_merge"] == "left_only"].drop(columns=["_merge"])
    else:
        df_base_sin = df_base_merge[base_cols].drop_duplicates()

    # =====================================================
    # ORDEN
    # =====================================================
    if "FECHA_FACTURACION_DT" in df_coincidencias.columns:
        df_coincidencias = df_coincidencias.sort_values(
            by=["SUMINISTRO_COMP", "PERIODO_ANIO_MES_BASE", "FECHA_FACTURACION_DT"],
            ascending=[True, True, True]
        )

    # =====================================================
    # GUARDAR
    # =====================================================
    with pd.ExcelWriter(ARCHIVO_SALIDA_CRUCE, engine="openpyxl") as writer:
        df_coincidencias.to_excel(writer, sheet_name="Coincidencias", index=False)
        df_base_sin.to_excel(writer, sheet_name="Base_Sin_Coincidencia", index=False)
        df_base_merge.to_excel(writer, sheet_name="Base_Suministros", index=False)

    ajustar_ancho_y_estilo_excel(ARCHIVO_SALIDA_CRUCE)

    print("=" * 80)
    print("CRUCE GENERADO")
    print("=" * 80)
    print(f"Archivo: {ARCHIVO_SALIDA_CRUCE}")
    print(f"Coincidencias: {len(df_coincidencias)}")
    print(f"Base sin coincidencia: {len(df_base_sin)}")


# =========================================================
# MAIN
# =========================================================

def main():
    os.makedirs(RUTA_OUTPUT, exist_ok=True)

    df_todos = consolidar_excels_individuales()
    generar_cruce_coincidencias(df_todos)

    print("=" * 80)
    print("PROCESO TERMINADO")
    print("=" * 80)
    print(f"Salida 1: {ARCHIVO_SALIDA_CONSOLIDADO}")
    print(f"Salida 2: {ARCHIVO_SALIDA_CRUCE}")


if __name__ == "__main__":
    main()

C:\Users\P760033411\AppData\Local\Temp\ipykernel_37124\870323247.py:228: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_todos = pd.concat(lista_todos, ignore_index=True, sort=False)


CONSOLIDADO GENERADO
Archivo: C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\BUSCADOR DE COMPENSACIONES OFICIO\OUT-PUT\01_CONSOLIDADO_EXCEL_INDIVIDUALES.xlsx
Filas hoja Todos_Consolidados: 19588
Filas hoja Solo_Compens_4Hrs: 389
CRUCE GENERADO
Archivo: C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\BUSCADOR DE COMPENSACIONES OFICIO\OUT-PUT\02_CRUCE_COINCIDENCIAS.xlsx
Coincidencias: 8544
Base sin coincidencia: 41
PROCESO TERMINADO
Salida 1: C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\BUSCADOR DE COMPENSACIONES OFICIO\OUT-PUT\01_CONSOLIDADO_EXCEL_INDIVIDUALES.xlsx
Salida 2: C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\BUSCADOR DE COMPENSACIONES OFICIO\OUT-PUT\02_CRUCE_COINCIDENCIAS.xlsx
